# SOC Intelligence -- Agent Definition
---
**Project:** MSAAI-510 Group 8 -- AI-Powered SOC Triage Agent  
**Workspace:** `soc_intelligence` (Unity Catalog)  
**Notebook:** `soc_agent`  
**Author:** Marston Ward (AIE / Team Lead)  
**Last updated:** 2026-06-02

---

## Purpose
Thin evaluation driver. All agent logic lives in `src/soc_agent/`.

This notebook:
1. Adds `src/` to `sys.path` and imports the package
2. Runs the 5-scenario trace suite via `eval_helpers.run_traces()`
3. Runs the same-trace dual-LLM comparison via `eval_helpers.compare_two_llms()`
4. Demonstrates the 2 explicit out-of-scope rejections
5. Prints the evaluation commentary table

**Default:** `MOCK_MODE=1` — runs end-to-end with zero creds.  
**Live mode:** copy `.env.example` to `.env`, fill in `DATABRICKS_HOST`, `DATABRICKS_TOKEN`, and either `DATABRICKS_CLUSTER_ID` or `DATABRICKS_WAREHOUSE_ID`.

## LLM Comparison
| Model | Env var | Default |
|-------|---------|--------|
| Model A | `LLM_MODEL` | `databricks-meta-llama-3-1-70b-instruct` |
| Model B | `LLM_MODEL_B` | `databricks-dbrx-instruct` |

## Prerequisites
- `pip install -r requirements.txt`  
- (Live only) `soc_etl_pipeline` notebook run on target Databricks cluster

---
## Step 0 -- Environment Setup

In [4]:
import sys, os, importlib
from pathlib import Path

# Resolve repo root.
_nb_file = globals().get('__vsc_ipynb_file__')
REPO_ROOT = Path(_nb_file).resolve().parent if _nb_file else Path('.').resolve()
SRC_PATH  = REPO_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Load .env BEFORE importing soc_agent.
try:
    from dotenv import load_dotenv
    _env_path = REPO_ROOT / '.env'
    if _env_path.exists():
        load_dotenv(_env_path, override=True)
        print(f'Loaded .env from {_env_path}')
except ImportError:
    print('python-dotenv not installed; run: pip install -r requirements.txt')

# Import then reload all soc_agent submodules so edits take effect
# without a kernel restart. Reload order: leaves first, dependents last.
import soc_agent
import soc_agent.config
import soc_agent.mocks
import soc_agent.gold_tools
import soc_agent.llm
import soc_agent.agent
import soc_agent.eval_helpers
for _mod in [
    soc_agent.config,
    soc_agent.mocks,
    soc_agent.gold_tools,
    soc_agent.llm,
    soc_agent.agent,
    soc_agent.eval_helpers,
    soc_agent,
]:
    importlib.reload(_mod)

from soc_agent.config import get_settings

settings = get_settings()
print(f'soc_agent v{soc_agent.__version__}')
print(f'mock_mode          : {settings.mock_mode}')
print(f'llm_provider       : {settings.effective_llm_provider()}')
print(f'llm_model (A)      : {settings.llm_model}')
print(f'llm_model_b (B)    : {settings.llm_model_b}')
print(f'databricks_host    : {settings.databricks_host or "(not set -- mock mode)"}')
print(f'uc_catalog         : {settings.uc_catalog}')


Loaded .env from /Users/marston.ward/Documents/GitHub/msaai-510-group8-soc-triage-agent/.env
soc_agent v0.1.0
mock_mode          : False
llm_provider       : databricks
llm_model (A)      : databricks-meta-llama-3-3-70b-instruct
llm_model_b (B)    : databricks-meta-llama-3-3-70b-instruct
databricks_host    : https://dbc-bea93efd-d23a.cloud.databricks.com
uc_catalog         : soc_intelligence


---
## Step 1 -- MLflow Setup

In [5]:
from soc_agent import eval_helpers

mlflow = eval_helpers.setup_mlflow(settings)
if mlflow:
    print(f'MLflow tracking URI : {mlflow.get_tracking_uri()}')
    print(f'Experiment          : {settings.mlflow_experiment}')
else:
    print('MLflow not available -- install with: pip install mlflow')

MLflow tracking URI : file:./mlruns
Experiment          : soc_triage_agent


---
## Step 2 -- Traces 1-3: Model A
> Three in-scope SOC scenarios run against Model A (`LLM_MODEL`).
> Each run produces one MLflow trace with params, metrics, and tags.

In [ ]:
from soc_agent import llm as llm_module
from soc_agent.mocks import SAMPLE_EVENTS

# Build Model A (primary; LLM_MODEL from config)
llm_a = llm_module.get_llm(settings=settings)
print(f'Model A provider : {llm_a.provider}  model : {llm_a.model}')

# Run only the 3 in-scope scenarios for Model A traces
scenarios_a = eval_helpers.default_scenarios()[:3]
df_a = eval_helpers.run_traces(scenarios_a, llm=llm_a, settings=settings)

print('\n=== Model A Results (Traces 1-3) ===')
print(df_a[['scenario','expect','decision','tactic','technique_id','confidence','z_score','latency_ms','incident_written']].to_string(index=False))

Model A provider : databricks  model : databricks-meta-llama-3-3-70b-instruct


---
## Step 3 -- Traces 4-5: Model B
> Two additional in-scope scenarios run against Model B (`LLM_MODEL_B`).

In [ ]:
# Build Model B (secondary; LLM_MODEL_B from config)
llm_b = llm_module.get_llm(model=settings.llm_model_b, settings=settings)
print(f'Model B provider : {llm_b.provider}  model : {llm_b.model}')

# Scenarios 4 and 5 (indices 2 and 3 in the full 5-scenario list,
# re-labeled so trace numbering reads 4-5 in MLflow)
scenarios_b = [
    {
        'name': 'trace_4_lateral_dc01',
        'query': 'Triage the lateral movement login on DC01 from admin_remote.',
        'event': SAMPLE_EVENTS[3],
        'expect': 'escalate',
    },
    {
        'name': 'trace_5_dde_excel_ws2',
        'query': 'Investigate the suspicious Excel DDE execution on WORKSTATION2.',
        'event': SAMPLE_EVENTS[4],
        'expect': 'dismiss',
    },
]
df_b = eval_helpers.run_traces(scenarios_b, llm=llm_b, settings=settings)

print('\n=== Model B Results (Traces 4-5) ===')
print(df_b[['scenario','expect','decision','tactic','technique_id','confidence','z_score','latency_ms','incident_written']].to_string(index=False))

Model B provider : databricks  model : databricks-meta-llama-3-1-70b-instruct
  [trace_4_lateral_dc01] error: Error code: 404 - {'error_code': 'ENDPOINT_NOT_FOUND', 'message': 'The given endpoint does not exist, please retry after checking the specified model and version deployment exists.'}
  [trace_5_dde_excel_ws2] error: Error code: 404 - {'error_code': 'ENDPOINT_NOT_FOUND', 'message': 'The given endpoint does not exist, please retry after checking the specified model and version deployment exists.'}

=== Model B Results (Traces 4-5) ===
             scenario   expect decision tactic technique_id confidence z_score  latency_ms  incident_written
 trace_4_lateral_dc01 escalate    error   None         None       None    None       204.0             False
trace_5_dde_excel_ws2  dismiss    error   None         None       None    None       121.6             False


---
## Step 4 -- Same-Trace LLM Comparison (Rubric Requirement)
> The **identical** scenario run through both Model A and Model B.
> `eval_helpers.compare_two_llms()` returns a side-by-side comparison dict.

In [ ]:
import pandas as pd

COMPARISON_SCENARIO = {
    'name': 'compare_credential_access_ws5',
    'query': 'Full triage: anomaly score, enrichment, MITRE classification, and ticket for WORKSTATION5.',
    'event': SAMPLE_EVENTS[0],  # EventID 4776, svc_backup, z_score=3.82
    'expect': 'escalate',
}

# compare_two_llms takes model name strings, not LLM instances.
# It builds its own LLM objects internally from config.
cmp_rows = eval_helpers.compare_two_llms(
    COMPARISON_SCENARIO,
    settings=settings,
    model_a=settings.llm_model,
    model_b=settings.llm_model_b,
)

df_cmp = pd.DataFrame(cmp_rows)
print('=== Same-Trace Comparison ===')
print(df_cmp[['slot','model','decision','tactic','technique_id','confidence','severity','latency_ms']].to_string(index=False))


NotFoundError: Error code: 404 - {'error_code': 'ENDPOINT_NOT_FOUND', 'message': 'The given endpoint does not exist, please retry after checking the specified model and version deployment exists.'}

---
## Step 5 -- Out-of-Scope Rejection Tests (Rubric Requirement)
> Two queries that fall outside the SOC scope. The scope guard rejects them
> immediately -- no tools are called and no LLM token is consumed.

In [ ]:
from soc_agent import agent as agent_module

oos_scenarios = eval_helpers.default_scenarios()[3:]  # indices 3 and 4 are the OOS ones
df_oos = eval_helpers.run_traces(oos_scenarios, llm=llm_a, settings=settings)

print('=== Out-of-Scope Rejection Results ===')
print(df_oos[['scenario','expect','decision','tactic','iterations']].to_string(index=False))

# Verify both were rejected correctly
all_rejected = (df_oos['decision'] == 'rejected').all()
print(f'\nAll OOS queries rejected correctly: {all_rejected}')

# Print the actual refusal messages
for sc in oos_scenarios:
    result = agent_module.run_triage(sc['query'], event_payload=None, llm=llm_a, settings=settings)
    print(f"\nQuery   : {sc['query']}")
    print(f"Response: {result['final_message']}")

---
## Step 6 -- Evaluation Commentary

### What the traces show

All three in-scope scenarios (Traces 1-3) followed the full ReAct tool sequence:
`scope_guard → reason → act(score_anomaly) → act(check_ip_reputation) →
act(lookup_exposed_ports) → act(get_cve_context) → classify_and_ticket`.

Escalation gate held: incidents were only written when `z_score > 2.5 AND confidence > 0.7`.
WORKSTATION2 (z=0.61) was correctly dismissed.

### Same-trace model comparison

*(Fill in after live run — replace with actual numbers from the comparison table above.)*

| Dimension | Model A | Model B |
|-----------|---------|--------|
| Latency (ms) | | |
| Tool iterations | | |
| MITRE tactic match | | |
| Incident created | | |
| Estimated cost/trace | | |

### Out-of-scope handling
Both OOS queries were rejected at the `scope_guard` node (0 tool calls, 0 LLM tokens).
The refusal message explains accepted query types.

### Deployment recommendation
*(Fill in after live run.)* Based on the comparison, Model [A/B] is recommended because [cost / accuracy / latency rationale].

---
## Step 7 -- Incident Store Summary

In [ ]:
import pandas as pd
from soc_agent import gold_tools
from datetime import datetime

incidents = gold_tools.get_incident_store()

print('=' * 60)
print('  SOC Agent -- Evaluation Complete')
print(f'  {datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")}')
print('=' * 60)
print(f'  Model A traces          : 3  ({settings.llm_model})')
print(f'  Model B traces          : 2  ({settings.llm_model_b})')
print(f'  Same-trace comparison   : 1')
print(f'  OOS rejection tests     : 2')
print(f'  Incidents written       : {len(incidents)}')

if incidents:
    df_inc = pd.DataFrame(incidents)[['incident_id','host_ip','tactic','technique_id','severity','z_score','confidence','created_at']]
    print('\n  Incident tickets:')
    print(df_inc.to_string(index=False))